# Processed data

This tutorial walks through every processed-data file shipped per session. Plotting and analysis examples live in [`example_analysis.ipynb`](example_analysis.ipynb); the maze graph helpers are covered in [`maze_intro.ipynb`](maze_intro.ipynb).

**Conventions** (also documented in the [README](../README.md)):

- All times are in **seconds from the start of the behavioural session** — cross-stream alignment is done at preprocessing time.
- All other quantities are in **SI units** unless stated otherwise.
- Tables are loaded as pandas DataFrames, arrays as numpy arrays, metadata as dicts.

In [1]:
from GridMaze.core.get_sessions import get_maze_sessions

with_lfp = True  # set False to skip the LFP files (they're the largest part of a session)

## Loading a session

`get_maze_sessions` returns one or more `MazeSession` objects. Each item in `with_data` becomes an attribute holding the loaded array/DataFrame; `has_data` lists which loads succeeded.

In [2]:
data_keys = [
    "trials_df",  # trials.htsv
    "events_df",  # events.htsv
    "spike_times",  # spikes.times.npy
    "spike_clusters",  # spikes.clusters.npy
    "cluster_metrics",  # clusters.metrics.htsv
    "tracking_df",  # frames.tracking.htsv
    "trajectories_df",  # frames.trajectories.htsv
    "trial_info_df",  # frames.trialInfo.htsv
]
if with_lfp:
    data_keys += ["lfp_times", "lfp_signal", "lfp_metrics"]

session = get_maze_sessions(
    subject_IDs=["m2"],
    maze_names=["maze_1"],
    days_on_maze=[10],
    with_data=data_keys,
)
print(session)


-MazeSession--------------------------------------------------
  Subject ID     : m2                      
  Maze Name      : maze_1                  
  Day on Maze    : 10                      
  Goal Subset    : all                     
  Date           : 2022-07-02              
---------------------------------------------------------------



## `trials_df` — per-trial behaviour

One row per trial. Each trial is: a goal is cued → the animal navigates to it → reward is delivered → ITI → next trial. Columns include:

- `trial` — trial number.
- `goal` — the goal tower active on that trial (e.g. `"E4"`).
- `n_error_pokes` — port pokes that weren't the goal.
- Event times (s): `cue`, `reward`, `end_reward_consumption`, `ITI_start`, `trial_end`.

In [3]:
session.trials_df.head(10)

trial goal errors     time                                            \
                         cue   reward end_reward_consumption ITI_start   
0     1   D3      9    7.007  116.027                120.140   120.649   
1     2   F7      2  124.650  153.033                154.613   155.122   
2     3   C7      0  161.123  170.887                176.207   176.716   
3     4   D6      3  182.717  273.489                274.447   274.957   
4     5   G1      5  279.958  333.167                337.863   338.373   
5     6   B4      3  345.374  393.558                394.384   394.893   
6     7   E3      0  402.894  416.290                417.171   417.681   
7     8   C5      0  421.682  424.089                424.707   425.217   
8     9   F2      0  431.218  440.683                441.396   441.905   
9    10   E6      0  446.906  465.110                465.816   466.326   

             
  trial_end  
0   124.650  
1   161.123  
2   182.717  
3   279.958  
4   345.374  
5   402.894  
6   421.682  
7   431.218  
8   446.906  
9   471.327

## `events_df` — every pyControl event

One row per pyControl event registered during the session — trial events (`cue`, `reward`, …) and hardware events (e.g. `"E6_in"` = poke into the `E6` port). Useful for fine-grained behavioural analyses such as inspecting error-poke during navigation.

In [4]:
session.events_df.head(10)

,type,name,time,duration,value
0,state,ITI,0.000,7.007,NaN
1,state,cue,7.007,109.020,NaN
2,print,NaN,7.013,NaN,Start trial - T#:1 S#:D3 sT#:7007
3,event,E6_in,14.032,0.748,NaN
4,event,D7_in,20.051,0.244,NaN
5,event,C6_in,26.824,0.476,NaN
6,event,F1_in,42.742,0.057,NaN
7,event,G7_in,85.980,1.809,NaN
8,event,G6_in,94.894,0.088,NaN
9,event,G6_out,95.111,NaN,NaN


## `spike_times` & `spike_clusters` — spike events

Two parallel numpy arrays of equal length:

- `spike_times` — time (s) of every detected spike.
- `spike_clusters` — the cluster ID that emitted each spike.

Cluster IDs index into `cluster_metrics`.

In [5]:
print(session.spike_times)
print(session.spike_clusters)

[ -47.06556324  -47.0647299   -47.0626632  ... 2415.62341898 2415.62378565
 2415.62435232]
[136  38  42 ...  34  15  16]


## `cluster_metrics` — per-cluster QC, region, and probe location

One row per cluster (a neuron or putative neuron) recorded in the session. Columns cover:

- **Quality metrics** — `firing_rate`, `presence_ratio`, `isi_violations_ratio`, `amplitude_median`, `amplitude_cutoff`, `sd_ratio`. See [SpikeInterface docs](https://spikeinterface.readthedocs.io/en/latest/modules/qualitymetrics.html) for what each one says about a cluster.
- **Classification** — booleans `single_unit`, `multi_unit`, `noise_unit`.
- **Contact** — `id`, `shank` (recordings are on 6-shank probes), and `x`/`y` position on the shank (µm).
- **`probe_depth`** — depth below brain surface where the cluster was recorded.
- **`tissue_sample`** — the probe was moved on a microdrive during the experiment; a tissue sample is one stable recording period before the next descent.
- **`voxel`** / **`region`** — Allen CCFv3 25 µm atlas voxel and region (name + abbreviation) for the cluster.

In [6]:
session.cluster_metrics.head(5)

cluster_ID quality_metrics                                      \
                 firing_rate presence_ratio isi_violations_ratio   
0          0        2.485524            1.0             1.489867   
1          1        2.565518            1.0             1.172192   
2          2        9.598969            1.0             0.095486   
3          3        8.885513            1.0             0.066861   
4          4        1.866681            1.0             0.038845   

                                              single_unit multi_unit  \
  amplitude_cutoff amplitude_median  sd_ratio                          
0         0.000826        78.780000  4.686416       False       True   
1         0.008078        19.695000  1.516020       False      False   
2         0.000037        76.049995  1.322952        True      False   
3         0.000196        74.100000  1.443254        True      False   
4         0.001197       169.650000  1.459052        True      False   

  noise_unit  ... tissue_sample probe_depth contact              voxel       \
              ...                             shank      x     y     x    y   
0      False  ...             B        1300       2  400.0  30.0   133  108   
1       True  ...             B        1300       2  400.0  30.0   133  108   
2      False  ...             B        1300       2  400.0  30.0   133  108   
3      False  ...             B        1300       2  400.0  30.0   133  108   
4      False  ...             B        1300       2  416.5  75.0   132  107   

        region                           
     z acronym                     name  
0  252     PL5  Prelimbic area, layer 5  
1  252     PL5  Prelimbic area, layer 5  
2  252     PL5  Prelimbic area, layer 5  
3  252     PL5  Prelimbic area, layer 5  
4  252     PL5  Prelimbic area, layer 5  

[5 rows x 21 columns]

## `tracking_df` — raw bodypart positions

One row per video frame. Columns are the `(x, y)` positions (m) of every bodypart tracked by SLEAP/DLC. Not quality-controlled — contains `NaN`s where a bodypart could not be estimated. Useful for pose-based analyses (MoSeq, behavioural clustering, …).

In [7]:
session.tracking_df.head(10)

head_front      head_mid           head_back               ear_L            \
           x   y         x         y         x         y         x         y   
0        NaN NaN  0.489365  0.502220  0.483139  0.505903  0.482787  0.508674   
1        NaN NaN  0.491436  0.499798  0.484940  0.502908  0.486652  0.506823   
2        NaN NaN  0.487993  0.497162  0.483041  0.501074  0.483549  0.503891   
3        NaN NaN  0.488832  0.495763  0.483072  0.499959  0.484442  0.502911   
4        NaN NaN  0.490155  0.496078  0.484136  0.500687  0.485352  0.503586   
5        NaN NaN  0.495935  0.489752  0.491310  0.496559  0.488812  0.495714   
6        NaN NaN  0.495229  0.490661  0.489878  0.497459  0.488994  0.496703   
7        NaN NaN  0.494711  0.487634  0.491615  0.493191  0.489963  0.493363   
8        NaN NaN  0.499232  0.486090  0.497142  0.492370  0.495119  0.491386   
9        NaN NaN  0.499035  0.488745  0.496890  0.495794  0.494942  0.493032   

      ear_R           body_front            body_mid           body_back  \
          x         y          x         y         x         y         x   
0  0.480684  0.500693   0.474669  0.510059  0.454935  0.514180  0.438195   
1  0.482538  0.498521   0.475340  0.505933  0.457557  0.511149  0.440269   
2  0.480413  0.496868   0.472075  0.503700  0.456401  0.510648  0.439202   
3  0.480312  0.495993   0.473784  0.503608  0.457216  0.509670  0.439486   
4  0.481442  0.496676   0.474572  0.504812  0.457158  0.511391  0.439269   
5  0.487809  0.492635   0.480108  0.500316  0.461428  0.509299  0.443185   
6  0.485957  0.493625   0.479184  0.501792  0.459234  0.510082  0.440935   
7  0.488305  0.489971   0.480264  0.499364  0.461406  0.507740  0.442921   
8  0.492660  0.490524   0.484909  0.498397  0.465029  0.507278  0.445010   
9  0.493282  0.492243   0.486033  0.500818  0.466122  0.509227  0.446735   

             
          y  
0  0.519283  
1  0.516859  
2  0.516078  
3  0.515608  
4  0.517789  
5  0.515093  
6  0.517575  
7  0.514028  
8  0.514077  
9  0.516130

## `trajectories_df` — QC'd centroid + head direction

One row per video frame. Quality-controlled fields:

- `head_direction` — degrees, with a flag indicating whether the value was interpolated during QC.
- `centroid_position` — `(x, y)` in metres, taken as the back of the head; same interpolation flag.
- `maze_position`:
    - `simple` — current node/edge on the [`simple_maze`](maze_intro.ipynb) (invalid transitions due to tracking errors corrected).
    - `skeleton` — current node/edge on the [`skeleton_maze`](maze_intro.ipynb) (see [maze intro notebook](maze_intro.ipynb) for info on maze representations).
- `time` — time (s) in session.

In [8]:
session.trajectories_df.head(10)

time head_direction              centroid_position            \
                     value interpolated                 x         y   
0 -6.767164     337.312370            0          0.483139  0.505903   
1 -6.750496     334.027132            0          0.484940  0.502908   
2 -6.733828     328.812234            0          0.483041  0.501074   
3 -6.717161     326.543193            0          0.483072  0.499959   
4 -6.700493     326.525941            0          0.484136  0.500687   
5 -6.683825     323.080193            0          0.491310  0.496559   
6 -6.667157     311.794868            0          0.489878  0.497459   
7 -6.650489     316.541399            0          0.491615  0.493191   
8 -6.633821     288.862301            0          0.497142  0.492370   
9 -6.617153     291.171536            0          0.496890  0.495794   

               maze_position           
  interpolated        simple skeleton  
0            0            C3     C3_C  
1            0            C3     C3_C  
2            0            C3     C3_C  
3            0            C3     C3_C  
4            0            C3     C3_C  
5            0            C3     C3_C  
6            0            C3     C3_C  
7            0            C3     C3_C  
8            0            C3     C3_C  
9            0            C3     C3_C

## `trial_info_df` — per-frame trial / phase / goal

One row per video frame, giving the trial number, navigational phase (`"navigation"`, `"reward_consumption"`, `"ITI"`), and active goal at that frame.

In [9]:
session.trial_info_df.iloc[1000:1010]

,trial,trial_phase,goal
1000,1.0,navigation,D3
1001,1.0,navigation,D3
1002,1.0,navigation,D3
1003,1.0,navigation,D3
1004,1.0,navigation,D3
1005,1.0,navigation,D3
1006,1.0,navigation,D3
1007,1.0,navigation,D3
1008,1.0,navigation,D3
1009,1.0,navigation,D3


> **Shared row index.** `tracking_df`, `trajectories_df`, and `trial_info_df` all have one row per video frame and share the same index, so you can use any of them as a boolean mask over the others (e.g. select trajectory rows from frames where `trial_info_df.trial == 42`).

## `lfp_times` — LFP timestamps

Time (s) for each column of `lfp_signal` (same alignment as everything else in the session).

In [10]:
if with_lfp:
    print(session.lfp_times)

[ -47.06579658  -47.0651299   -47.06446323 ... 2415.62431899 2415.62498566
 2415.62565234]


## `lfp_signal` — LFP signal (µV)

`(n_channels, n_samples)` numpy array containing the LFP from a subset of probe channels. The channel order matches `lfp_metrics` row order.

In [11]:
if with_lfp:
    print(session.lfp_signal.shape)  # (n_channels, n_samples)
    print(session.lfp_signal)

(3693990, 34)
[[  0.19494629   0.          -7.2148438  ... -36.84375    -39.59375
   -9.5546875 ]
 [ -8.578125   -14.4296875  -21.0625     ... -56.15625    -56.15625
   14.0390625 ]
 [-25.15625    -49.125      -47.1875     ... -84.625      -76.4375
   14.234375  ]
 ...
 [-10.3359375  -32.375       -2.1445312  ... -14.4296875  -12.28125
   -3.3144531 ]
 [ -8.1875     -21.84375      3.5097656  ...  -2.7304688   -3.9003906
    3.7050781 ]
 [ -7.8007812   -6.6289062    2.9257812  ...   5.65625      0.97509766
    3.7050781 ]]


## `lfp_metrics` — per-channel QC, region, and sampling rate

One row per channel of `lfp_signal`. Contains:

- `tissue_sample`, `probe_depth`.
- Contact `id` and `shank`.
- `voxel` / `region` — Allen CCFv3 25 µm atlas location.
- `qc` — `"good"`, `"bad"`, or `"dead"`.
- `sampling_rate` (Hz).

Useful for selecting channels by shank, depth, or region before working with `lfp_signal`.

In [12]:
if with_lfp:
    display(session.lfp_metrics.head(10))

tissue_sample probe_depth contact                      voxel            \
                                 id shank       x      y     x    y    z   
0             B        1300       5     5  1000.0   60.0   155  107  253   
1             B        1300       6     4   800.0  150.0   148  104  253   
2             B        1300       7     3   600.0   30.0   141  108  252   
3             B        1300       9     2   400.0   30.0   133  108  252   
4             B        1300      13     0     0.0    0.0   163  110  253   
5             B        1300      14     1   200.0   90.0   126  106  251   
6             B        1300      15     1   200.0   30.0   126  108  251   
7             B        1300      16     0     0.0   90.0   163  106  253   
8             B        1300      17     5  1000.0  120.0   155  105  253   
9             B        1300      18     3   600.0   90.0   141  106  252   

   region                                                 contact  \
  acronym                                            name      qc   
0     PL5                         Prelimbic area, layer 5    good   
1     PL5                         Prelimbic area, layer 5    good   
2     PL5                         Prelimbic area, layer 5    good   
3     PL5                         Prelimbic area, layer 5    good   
4   ACAv5  Anterior cingulate area, ventral part, layer 5    good   
5     PL5                         Prelimbic area, layer 5    good   
6     PL5                         Prelimbic area, layer 5    good   
7   ACAv5  Anterior cingulate area, ventral part, layer 5    dead   
8     PL5                         Prelimbic area, layer 5    good   
9     PL5                         Prelimbic area, layer 5    good   

  sampling_rate  
                 
0          1500  
1          1500  
2          1500  
3          1500  
4          1500  
5          1500  
6          1500  
7          1500  
8          1500  
9          1500

## `session_info` — static session metadata

A plain dict carrying everything that's fixed for the session: `subject_ID`, `session_type`, `session_date`, `experimental_day`, `maze_name`, `maze_structure`, `day_on_maze`, `goal_subset`, `goals`, `reward_size`, `probe_depth`, `tissue_sample`.

Each key is also unpacked as an attribute on the session object (e.g. `session.maze_name`).

In [13]:
session.session_info

{'subject_ID': 'm2',
 'session_type': 'maze',
 'session_date': '2022-07-02',
 'experimental_day': 10,
 'maze_name': 'maze_1',
 'maze_structure': ['A1-A2',
  'A3-A4',
  'A4-A5',
  'A5-A6',
  'A6-A7',
  'A2-B2',
  'A3-B3',
  'A5-B5',
  'A7-B7',
  'B4-B5',
  'B6-B7',
  'B1-C1',
  'B2-C2',
  'B3-C3',
  'B6-C6',
  'C1-C2',
  'C2-C3',
  'C3-C4',
  'C4-C5',
  'C5-C6',
  'C6-C7',
  'C2-D2',
  'C5-D5',
  'C7-D7',
  'D1-D2',
  'D3-D4',
  'D4-D5',
  'D6-D7',
  'D1-E1',
  'D2-E2',
  'D3-E3',
  'D4-E4',
  'D5-E5',
  'D6-E6',
  'E2-F2',
  'E3-F3',
  'E5-F5',
  'E6-F6',
  'E7-F7',
  'F1-F2',
  'F2-F3',
  'F4-F5',
  'F6-F7',
  'F2-G2',
  'F5-G5',
  'F6-G6',
  'G1-G2',
  'G2-G3',
  'G3-G4',
  'G4-G5',
  'G5-G6',
  'G6-G7'],
 'day_on_maze': 10,
 'goal_subset': 'all',
 'goals': ['A2',
  'A3',
  'A6',
  'B4',
  'B5',
  'B6',
  'C1',
  'C3',
  'C5',
  'C7',
  'D2',
  'D3',
  'D6',
  'E1',
  'E3',
  'E4',
  'E5',
  'E6',
  'F2',
  'F4',
  'F7',
  'G1',
  'G4',
  'G7'],
 'reward_size': '8uL',
 'probe_depth':